In [16]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [17]:
df = pd.read_csv('marriage_longevity_master.csv')
df.head()

,marriage_id,marriage_number,age_at_marriage,age_gap_years,education_level,household_income_usd,financial_stress,both_employed,cohabited_before,premarital_counseling,...,contempt,defensiveness,stonewalling,repair_attempt_success,positive_negative_ratio,conflict_frequency_weekly,shared_activities_weekly,years_married,years_to_divorce,divorced
0,MAR000000,1,36,6,less_than_hs,68600.0,5.9,1,1,0,...,7.7,5.8,8.1,4.1,2.22,3.0,4.0,22.0,NaN,0
1,MAR000001,2,59,1,some_college,328800.0,0.0,1,1,0,...,1.7,6.8,3.0,5.1,11.99,1.0,1.0,15.5,NaN,0
2,MAR000002,2,44,1,high_school,139900.0,6.1,0,0,1,...,6.5,4.2,4.8,5.7,2.27,4.0,2.0,7.2,NaN,0
3,MAR000003,1,25,2,bachelors,97600.0,5.9,1,1,1,...,0.0,4.1,0.9,8.7,6.24,0.0,4.0,13.5,NaN,0
4,MAR000004,2,29,3,graduate,98200.0,4.0,0,0,1,...,4.4,4.6,9.6,8.9,2.59,4.0,5.0,7.5,7.5,1


In [18]:
df = df.drop(columns=['years_married', 'years_to_divorce', 'marriage_id', 'marriage_number'], axis=1)

In [19]:
df['total_negativity'] = df['contempt'] + df['stonewalling'] + df['defensiveness'] + df['criticism']

In [20]:
df = df.drop(columns=['contempt', 'stonewalling', 'defensiveness', 'criticism'], axis=1)

In [21]:
df['income_per_person'] = df['household_income_usd'] / (df['n_children'] + 2)

In [22]:
df['conflict_x_repair'] = (df['conflict_frequency_weekly'] * df['repair_attempt_success'])

In [23]:
df = df.drop(columns=['conflict_frequency_weekly', 'repair_attempt_success'], axis=1)

In [24]:
df.head()

,age_at_marriage,age_gap_years,education_level,household_income_usd,financial_stress,both_employed,cohabited_before,premarital_counseling,child_before_marriage,n_children,religious_attendance,positive_negative_ratio,shared_activities_weekly,divorced,total_negativity,income_per_person,conflict_x_repair
0,36,6,less_than_hs,68600.0,5.9,1,1,0,0,2,never,2.22,4.0,0,26.4,17150.000000,12.3
1,59,1,some_college,328800.0,0.0,1,1,0,0,3,never,11.99,1.0,0,11.7,65760.000000,5.1
2,44,1,high_school,139900.0,6.1,0,0,1,0,1,never,2.27,2.0,0,20.7,46633.333333,22.8
3,25,2,bachelors,97600.0,5.9,1,1,1,0,2,rarely,6.24,4.0,0,8.7,24400.000000,0.0
4,29,3,graduate,98200.0,4.0,0,0,1,0,2,rarely,2.59,5.0,1,24.0,24550.000000,35.6


In [26]:
df = pd.get_dummies(data=df, columns=['education_level', 'religious_attendance'], dtype=int, drop_first=True)

In [28]:
print(df['age_at_marriage'].skew())
print(df['household_income_usd'].skew())

0.7365120894777876
2.0508142446591577


In [43]:
df_cleaned = df.copy()

In [44]:
df_cleaned['household_income_usd'] = np.log1p(df['household_income_usd'])

In [45]:
df_cleaned['household_income_usd'].skew()

np.float64(-0.02303011574926625)

In [46]:
df_cleaned['positive_negative_ratio'] = np.log1p(df['positive_negative_ratio'])

In [47]:
df_cleaned['positive_negative_ratio'].skew()

np.float64(0.3893268843106309)

In [48]:
df_cleaned['age_gap_years'] = np.sqrt(df['age_gap_years'])

In [49]:

df_cleaned['age_gap_years'].skew()

np.float64(-0.3580799297073255)

In [50]:
X = df_cleaned.drop(columns=['divorced'], axis=1)
y = df_cleaned['divorced']

In [51]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [55]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
n_columns = [ 'conflict_x_repair','total_negativity','income_per_person', 
            'age_at_marriage', 'age_gap_years', 'household_income_usd', 'financial_stress', 
            'positive_negative_ratio','shared_activities_weekly'] # 'conflict_x_repair','total_negativity','income_per_person',
X_train[n_columns] = scaler.fit_transform(X_train[n_columns])
X_test[n_columns] = scaler.transform(X_test[n_columns])

In [56]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

In [58]:
model.fit(X_train, y_train)

LogisticRegression()

In [59]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [60]:
y_pred = model.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred)
# accuracy : 0.72 

0.7202222222222222